# Milestone 022 on Colab

Use this notebook after switching the Colab runtime to `TPU`. The notebook clones the repo, installs TPU JAX, downloads the public tokenizer + token shards from Hugging Face into `/content`, runs `experiments/022_tpu_fineweb_edu_scaling_baseline.py`, displays the latest artifact, and copies a zip of the latest run into Google Drive.


In [ ]:
!git clone https://github.com/marcoshernanz/llm-lab.git /content/llm-lab
%cd /content/llm-lab
!git checkout milestone-022
!python -m pip install -q -U pip
!python -m pip install -q -U "jax[tpu]" flax optax numpy pyarrow datasets huggingface-hub equinox

In [ ]:
"""Download the public tokenizer and shard files from Hugging Face."""

from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "marcoshernanz/llm-lab-fineweb-edu-sample10bt-bpe-16384"
LOCAL_DATA_ROOT = Path("/content/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
LOCAL_TOKENIZER_PATH = LOCAL_DATA_ROOT / "fineweb_edu_sample10bt_bpe_16384.json"

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=str(LOCAL_DATA_ROOT),
    allow_patterns=["*.json", "*.npy"],
    token=False,
)

print("local_dataset_root=", LOCAL_DATA_ROOT)
print("local_tokenizer_path=", LOCAL_TOKENIZER_PATH)
print("downloaded_files=")
for path in sorted(LOCAL_DATA_ROOT.iterdir()):
    print("  ", path.name)


In [ ]:
!PYTHONPATH=/content/llm-lab python experiments/022_tpu_fineweb_edu_scaling_baseline.py \
  --token-shard-root /content/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384 \
  --tokenizer-path /content/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384/fineweb_edu_sample10bt_bpe_16384.json \
  --max-train-shards 10 \
  --train-steps 50000 \
  --train-chunk-length 100 \
  --batch-size 128

In [ ]:
"""Display the newest loss curve written by the milestone-022 run."""

from IPython.display import SVG, display
from pathlib import Path

run_dirs = sorted(Path("/content/llm-lab/artifacts/experiments/022_tpu_fineweb_edu_scaling_baseline").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No milestone-022 artifact directory was created.")

latest_run_dir = run_dirs[-1]
print("latest_run_dir=", latest_run_dir)
display(SVG(filename=str(latest_run_dir / "loss_curve.svg")))
print((latest_run_dir / "sample.txt").read_text(encoding="utf-8"))


In [ ]:
"""Zip the latest run for one experiment and copy the zip into Google Drive."""

from google.colab import drive
from pathlib import Path
import shutil

EXPERIMENT_NAME = "022_tpu_fineweb_edu_scaling_baseline"
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/llm-lab")

drive.mount("/content/drive")

runs_root = Path("/content/llm-lab/artifacts/experiments") / EXPERIMENT_NAME
run_dirs = sorted(path for path in runs_root.glob("*") if path.is_dir())
if not run_dirs:
    raise FileNotFoundError(f"No runs found under {runs_root}")

latest_run_dir = run_dirs[-1]
archive_base = latest_run_dir.parent / latest_run_dir.name
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(latest_run_dir.parent),
        base_dir=latest_run_dir.name,
    )
)

drive_target = DRIVE_OUTPUT_DIR / archive_path.name
drive_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive_path, drive_target)

print("latest_run_dir=", latest_run_dir)
print("archive_path=", archive_path)
print("copied_to=", drive_target)
